# 🌍 GeoIntel AI — Phase 1: The Brain

**What this is:** The simplest version of our geopolitics AI agent. We send a country name to Claude, and it writes a full geopolitical briefing from its own knowledge.

**No web search yet** — that's Phase 2. Right now we're getting the skeleton right: the structure, the system prompt, and the output format.

---

### What you're learning in this notebook:
1. How to call the Claude API with the Anthropic Python SDK
2. What a "system prompt" is and why it matters
3. How to get structured output from an LLM

---

## Step 1: Install & Import

The `anthropic` package is the official Python SDK for calling the Claude API.
Think of it as the "phone line" between your code and Claude's brain.

In [ ]:
# Run this once to install the SDK
# ! pip install anthropic

import anthropic
import os

## Step 2: Set Your API Key

Your API key is like a password that lets your code talk to Claude.
Get yours at: https://console.anthropic.com/settings/keys

⚠️ **NEVER hardcode your key in a notebook you share.** Use environment variables.

If you already set it in PowerShell with `:ANTHROPIC_API_KEY="sk-ant-..."`, 
Jupyter will pick it up automatically. Otherwise, uncomment the line below and paste it temporarily.

In [ ]:
# Option 1: If you set it in PowerShell already, this just works:
client = anthropic.Anthropic()

# Option 2: If that doesn't work, uncomment this and paste your key (delete before sharing!):
# client = anthropic.Anthropic(api_key="sk-ant-your-key-here")

print("✅ Client created successfully!")

## Step 3: The System Prompt — The Most Important Part

This is the **"job description"** you give to Claude. It controls everything about the output.

Think of it this way:
- **Bad system prompt** = hiring an intern and saying "write something about a country"
- **Good system prompt** = hiring an analyst and saying "produce a structured briefing with these 7 sections, in this tone, with this format"

**This is prompt engineering.** Change this text and the entire output changes.

In [ ]:
SYSTEM_PROMPT = """You are GeoIntel AI, a geopolitical research analyst.

When given a country name, produce a comprehensive geopolitical briefing 
with the following sections:

1. **Country Overview** — Official name, capital, population, region, 
   government type, current head of state/government.

2. **Historical Background** — Key historical events that shaped the 
   country's current political landscape. Focus on the last 100 years.
   Cover independence, major wars, revolutions, regime changes.

3. **Current Political Landscape** — Current government structure, 
   ruling party/coalition, major opposition, recent elections, 
   political stability assessment.

4. **Economy** — GDP, major industries, economic challenges, 
   trade relationships, sanctions if any.

5. **Foreign Relations & Alliances** — Key allies, adversaries, 
   regional role, membership in international organisations 
   (NATO, EU, BRICS, etc.), ongoing disputes.

6. **Military & Security** — Military strength overview, 
   nuclear status, active conflicts, internal security threats.

7. **Key Issues to Watch** — 2-3 current developments that could 
   shift the country's geopolitical position.

FORMAT RULES:
- Write in a professional but accessible tone (think: BBC World Service)
- Use bullet points for quick-scan readability
- Keep each section concise — quality over quantity
- End with a 2-sentence "Bottom Line" summary

NOTE: In this version you're working from your training knowledge only.
Be transparent about what you know and flag anything that may be outdated.
"""

print("✅ System prompt loaded!")
print(f"📏 Prompt length: {len(SYSTEM_PROMPT)} characters")

## Step 4: The API Call — Where the Magic Happens

This function sends your country name to Claude and gets back a briefing.

Key things to notice:
- **`model`** — which Claude model to use. `claude-sonnet-4-6` is a good balance of smart + affordable
- **`max_tokens`** — maximum length of the response (4096 ≈ ~3000 words)
- **`system`** — the system prompt (job description)
- **`messages`** — the conversation. Right now it's just one message from the user

Right now this is a **"one-shot" call** — one question, one answer. 
In Phase 2, we'll turn this into an **agent loop** where Claude calls tools and thinks multiple times.

In [ ]:
def generate_briefing(country: str) -> str:
    """Send a country name to Claude and get a geopolitical briefing back."""
    
    print(f"🌍 Generating briefing for: {country}")
    print("⏳ Thinking...
")
    
    # THE API CALL — this is the core of everything
    message = client.messages.create(
        model="claude-sonnet-4-6",        # Good balance of quality + cost
        max_tokens=4096,                   # Max response length
        system=SYSTEM_PROMPT,              # The "job description"
        messages=[                         # The conversation
            {
                "role": "user",
                "content": f"Generate a full geopolitical briefing for: {country}"
            }
        ]
    )
    
    # The API returns a structured object.
    # The actual text is inside: message.content[0].text
    return message.content[0].text

print("✅ Function ready!")

## Step 5: Run It! 🚀

Change the country name below and run the cell. Try different countries 
to see how the briefing changes.

**Try these:** Turkey, Iran, Taiwan, Ukraine, India, Brazil, Nigeria

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 👇 CHANGE THIS TO ANY COUNTRY YOU WANT
country = "Turkey"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

briefing = generate_briefing(country)

In [ ]:
# Display the briefing with nice formatting
from IPython.display import Markdown, display
display(Markdown(briefing))

## Step 6: Save the Briefing

This saves the output as a markdown file you can open, share, or reference later.

In [ ]:
# Save to a markdown file
filename = f"briefing_{country.lower().replace(' ', '_')}.md"
with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# GeoIntel AI Briefing: {country}

")
    f.write(briefing)

print(f"✅ Briefing saved to: {filename}")

---

## ✅ Phase 1 Complete! What you just learned:

| Concept | What it means | Where it is in the code |
|---------|--------------|------------------------|
| **API Client** | The connection to Claude | `client = anthropic.Anthropic()` |
| **System Prompt** | The "job description" that shapes output | `SYSTEM_PROMPT` variable |
| **Messages** | The conversation history | `messages=[{"role": "user", ...}]` |
| **Model Selection** | Which Claude brain to use | `model="claude-sonnet-4-6"` |
| **One-shot call** | Single question → single answer | `client.messages.create()` |

## 🔜 Phase 2: Adding Web Search (The Agent)

In the next notebook, we'll add **Tavily web search** as a tool that Claude 
can call. This is where it goes from "chatbot" to **"agent"** — Claude will 
decide what to search, read the results, and write the briefing with real 
sources and citations.